In [ ]:
## python modules used within this notebook
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import sys
import mynumerics as mn
import units
import HHG
from IPython.display import display, Markdown
from IPython.display import HTML


matplotlib.rcParams['animation.embed_limit'] = 200.

## TDSE with a custom input

We show the interface for the TDSE solver accessed directly through Python. This notebook shows some basic characteristics of the TDSE result.


First, we import the compiled dynamical library and its Pythonic wrapper:

In [ ]:
from PythonTDSE import *

# Compiled dynamic C library
path_to_DLL = os.path.join(os.environ['TDSE_1D_BUILD'],'libsingleTDSE.so')
DLL = TDSE_DLL(path_to_DLL)

### Define the custom input field & numerical parameters

Here we define the input parameters for the CTDSE solver and the initial pulse. We show an example of a chirped pulse with a $\sin^2$-envelope. The field is then given by

$$ \mathcal{E}(t) = \mathcal{E}_0 \sin^2 \left( \frac{t}{T_{\text{envelope}}} \right) \cos \left(\omega_0 t + \omega_c t^2 \right) \,.$$

(Note that the instantaneous frequency is then $\omega_i(t) = \omega_0 + 2\omega_c t$. This means that $\omega_0$ cannot be taken as the central frequency, the frequency at the peak of the pulse is $\omega_i(\pi T_{\text{envelope}}/2) = \omega_0 + \pi \omega_c T_{\text{envelope}}$.)

In [ ]:
omega0 = mn.ConvertPhoton(800e-9,'lambdaSI','omegaau')
chirp = 0*2e-4
# E_0 = 0.15   # peak electric field amplitude
Intensity_H_max = 30.
E_0 = np.sqrt(HHG.ComputeInvCutoff_gas(Intensity_H_max,omega0,'Ar'))  # peak electric field amplitude

T0 = mn.ConvertPhoton(omega0,'omegaau','T0au') # the duration of the reference cycle
T_max = 20*T0 # total pulse duration expressed in the number of the reference cycles
N_t = 25000  # # of points for field construction (not for TDSE)

# Construct the field
tgrid = np.linspace(0, T_max, N_t)
E = E_0* (np.sin(np.pi*tgrid/T_max)**2) *np.cos(omega0*tgrid + chirp*(tgrid)**2)


# Create instance of input structure
inputs = inputs_def()

# Set the inputs for the TDSE solver
trg_a = HHG.soft_Coulomb_a['Ar'] # soft-Coulomb-potnetial parameter from HHG module 
x_int = 2. # "The volume of the atom for volumetric ionisation"
inputs.init_default_inputs(
            Eguess   = -HHG.Ip_list['Ar'] ,       # ionisation potential from the HHG module 
            trg_a    = trg_a, 
            dt       = 0.125 ,
            dx       = 0.4 ,
            num_r    = 16000, # 300 with the absorber
            writewft = 1 ,
            tprint   = 1. ,
            x_int    = x_int,
            absorber = {'type'  : 0}
            # absorber = {'type'  : 1,
            #             'x_cap' : 50., # a.u.
            #             'alpha' : 0.001}  # a.u.)
            )
# Note: Parameters currently needs to be fixed for the gauge-invariant energetic analysis (gas & some of numerics for the same ensemble of bound states)

The parameters are fixed to match [the energetic analysis](#invariant_energy_distribution).

### Pipeline to execute the TDSE computation

In [ ]:
inputs.init_time_and_field(DLL, E = E, t = tgrid) # set our electric field as the input
DLL.init_GS(inputs)                               # create the C-types input for the C-library
output = outputs_def()                            # prepare the structure that holds the TDSE outputs 
DLL.call1DTDSE(inputs, output)                    # run TDSE

### Obtain detailed analyses and visualisation
Here we specify some parameters for various analyses and plotting

In [ ]:
H_max_plot = 60 # [harmonic order]

# Turn on-off various plots
plot_expval                = True
plot_sourceterm            = True
plot_volumetric_ionisation = True
plot_projected_ionisation  = True
plot_wavefunction          = True

# plot time in femtoseconds
time_fs = True


tgrid = 1e15*mn.ConvertPhoton(1.,'T0au','T0SI')*output.get_tgrid() if time_fs else output.get_tgrid()

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    tgrid,
    output.get_Efield() / np.max(np.abs(output.get_Efield())),
    label='Electric field'
)

if plot_sourceterm:
    ax.plot(
        tgrid,
        output.get_sourceterm() / np.max(np.abs(output.get_sourceterm())),
        label='Source term'
    )

if plot_expval:
    ax.plot(
        tgrid,
        output.get_expval() / np.max(np.abs(output.get_expval())),
        label='Expectation value'
    )

if plot_volumetric_ionisation:
    ax.plot(
        tgrid,
        1.-output.get_PopInt(),
        label=f'$P_{{\\text{{ion}}}}$ using $x_{{\\text{{int}}}}={x_int:.1f}$'
    )

if plot_projected_ionisation:
    ax.plot(
        tgrid,
        1.-output.get_PopTot(),
        label = r'$P_{\text{ion}}$ using $\left\langle \psi(t) | \psi_{\text{GS}} \right\rangle_{\text{inv}}$'
    )

# data = output.get_expval() - output.get_Efield()
# ax.plot(
#     tgrid,
#     data / np.max(data),
#     label=r'$\left\langle \frac{\partial^2 j}{\partial t^2} \right\rangle - E$'
# )

ax.set_xlim(tgrid[[0, -1]])
ax.set_ylim(-1, 1)
if time_fs: ax.set_xlabel(r'$t~[\mathrm{fs}]$')
else:       ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax.set_ylabel('Normalised value')
ax.legend()

fig.tight_layout()
plt.show()

omega_max_plot = omega0*H_max_plot
ogrid = output.get_omegagrid()
ko_max = mn.FindInterval(ogrid, omega_max_plot)

energy_eV = mn.ConvertPhoton(
    ogrid[:ko_max],
    'omegaau',
    'eV'
)

ogrid = output.get_omegagrid()
ko_max = mn.FindInterval(ogrid, omega_max_plot)

fig, ax = plt.subplots(figsize=(8, 5))

ax.semilogy(
    ogrid[:ko_max] / omega0,
    np.abs(output.get_Fsourceterm())[:ko_max],
    label='Source-term spectrum'
)

ax.set_xlim((ogrid[0] / omega0, ogrid[ko_max - 1] / omega0))
ax.set_xlabel(r'Harmonic order $\omega / \omega_0$')
ax.set_ylabel(
    r'$|(\partial \hat{\jmath}/\partial t)(\omega)|'
    r'~[\mathrm{arb.~u.}]$'
)
ax.legend()

fig.tight_layout()
plt.show()


# -------------------------------------------------------------------------
# 3. Wavefunction
# -------------------------------------------------------------------------

if plot_wavefunction:
    # Optional spatial filtering of the wavefunction plot
    filter_xgrid = False  # False: full x_grid; True: restrict the plotted range
    x_max_plot = 250.1    # [a.u.], used only when filter_xgrid is True
    
    # Optional lower-value filtering of the wavefunction
    filter_wavefunction_min = False
    wavefunction_min = 1e-8
    wavefunction_max = 0.5
    
    # Load the wavefunction
    t_psi, x_grid, wavefunction = output.get_wavefunction(
        inputs,
        grids=True
    )
    if time_fs: t_psi *= 1e15*mn.ConvertPhoton(1.,'T0au','T0SI')
    
    
    # Figure size in inches: (width, height)
    figsize_wavefunction = (8, 5.5)
    
    
    # Select the spatial range
    if filter_xgrid:
        x_range = np.abs(x_grid) < x_max_plot
    else:
        x_range = slice(None)
    
    x_plot = x_grid[x_range]
    
    
    # Prepare the wavefunction data
    psi_plot = np.abs(wavefunction).T[x_range]
    
    
    # Optionally replace values below wavefunction_min
    if filter_wavefunction_min:
        psi_display = np.maximum(
            psi_plot,
            wavefunction_min
        )
    else:
        psi_display = psi_plot
    
    
    # Logarithmic colour normalisation
    wavefunction_norm = colors.LogNorm(
        vmin=wavefunction_min,
        vmax=wavefunction_max
    )
    
    
    fig3, ax3 = plt.subplots(
        figsize=figsize_wavefunction,
        layout='constrained'
    )
    
    pc3 = ax3.pcolormesh(
        t_psi,
        x_plot,
        psi_display,
        cmap='jet',
        norm=wavefunction_norm,
        shading='auto'
    )
    
    if time_fs: ax3.set_xlabel(r'$t~[\mathrm{fs}]$')
    else:       ax3.set_xlabel(r'$t~[\mathrm{a.u.}]$')
    ax3.set_ylabel(r'$x~[\mathrm{a.u.}]$')
    
    ax3.set_xlim(
        np.min(t_psi),
        np.max(t_psi)
    )
    
    ax3.set_ylim(
        np.min(x_plot),
        np.max(x_plot)
    )
    
    
    # Horizontal colour bar below the plot
    cbar = fig3.colorbar(
        pc3,
        ax=ax3,
        orientation='horizontal',
        location='bottom',
        pad=0.08,
        shrink=0.8,
        aspect=45
    )
    
    cbar.set_label(r'$|\psi|~[\mathrm{a.u.}]$')
    
    plt.show()